# N-step Actor-Critic: stabilize learning with rollouts

This notebook is a more advanced implementation of Actor-Critic. Start with the [vanilla one-step Actor-Critic notebook](actor_critic.ipynb) to see the canonical online update before studying the rollout machinery introduced here.

Like the vanilla version, this implementation combines a stochastic **actor** $\pi_\theta(a\mid s)$ with a **critic** $V_\phi(s)$. Instead of updating from one transition at a time, it collects fixed-length rollouts and uses the critic to bootstrap an n-step return:

$$G_t^{(n)}=r_{t+1}+\gamma r_{t+2}+\cdots+\gamma^n V_\phi(s_{t+n}),$$

The estimate $\hat A_t=G_t^{(n)}-V_\phi(s_t)$ is the multi-step counterpart of the vanilla notebook's one-step TD error. Multi-step returns propagate reward information across the rollout instead of requiring an already-accurate critic at every next state. Fixed-length batches, advantage normalization, entropy regularization, gradient clipping, and Huber value loss make this implementation more sophisticated than the intentionally minimal vanilla example.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

ENV_ID = "CartPole-v1"
TOTAL_TIMESTEPS = 20_000
N_STEPS = 64
ACTOR_LEARNING_RATE = 3e-4
CRITIC_LEARNING_RATE = 1e-3
GAMMA = 0.99
ENTROPY_COEFFICIENT = 1e-3
MAX_GRAD_NORM = 1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = gym.make(ENV_ID)
observation_dim = int(np.prod(env.observation_space.shape))
action_dim = env.action_space.n
print(f"Observation size: {observation_dim}; actions: {action_dim}; device: {device}")

## 1. Build separate actor and critic networks

The actor outputs one logit per discrete action. A categorical distribution turns the logits into a stochastic policy. The critic outputs one scalar: its estimate of the discounted return from the observation. This separation matches the [vanilla implementation](actor_critic.ipynb), making it easier to focus on the more advanced learning signal and update cadence introduced below.

In [ ]:
def make_network(output_dim):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(observation_dim, 128),
        nn.ReLU(),
        nn.Linear(128, 128),
        nn.ReLU(),
        nn.Linear(128, output_dim),
    ).to(device)


actor = make_network(output_dim=action_dim) # Output a logit per action
critic = make_network(output_dim=1) # Output state value
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=ACTOR_LEARNING_RATE)
critic_optimizer = torch.optim.Adam(
    critic.parameters(), lr=CRITIC_LEARNING_RATE
)


def action_distribution(observations):
    observations = torch.as_tensor(
        observations, dtype=torch.float32, device=device
    )
    logits = actor(observations)
    return torch.distributions.Categorical(logits=logits)


def select_action(observation, deterministic=False):
    with torch.no_grad():
        distribution = action_distribution(observations=np.atleast_2d(observation))
        action = (
            distribution.probs.argmax(dim=-1)
            if deterministic
            else distribution.sample()
        )
    return int(action.item())

## 2. Construct the n-step learning signal

The vanilla notebook uses one reward and one next-state value to form each TD target. Here, returns are accumulated backward through an entire rollout and bootstrap from the critic at the rollout boundary. Gymnasium distinguishes `terminated` from `truncated`: a true MDP termination sets the bootstrap value to zero, while a time-limit truncation uses $V(s_{t+1})$. Both stop the backward recursion so rewards from a reset episode never leak into the preceding one.

The returns and advantages are detached. They are learning signals, so backpropagation must not flow through the bootstrap value or from the actor loss into the critic.

### Negative log-likelihood loss

The actor weights the sampled action's negative log-likelihood by its detached advantage:

$$L_{\text{policy}}=-\frac{1}{N}\sum_t \hat{A}_t \log \pi_\theta(a_t\mid s_t)$$

A positive advantage increases the action's probability; a negative one decreases it. Standardizing advantages within each fixed-length rollout reduces gradient-scale variation without changing their ordering.

### Entropy loss

$$H(\pi_\theta)=-\sum_a \pi_\theta(a\mid s)\log \pi_\theta(a\mid s)$$

$$L_{\text{actor}}=L_{\text{policy}}-\beta H(\pi_\theta)$$

Entropy measures how spread out the action probabilities are. Subtracting it rewards exploration and discourages the policy from becoming deterministic too early.

In [ ]:
def update(
    observations, actions, rewards, next_observations, terminated, episode_ends
):
    observations = torch.as_tensor(
        np.asarray(observations), dtype=torch.float32, device=device
    )
    actions = torch.as_tensor(actions, dtype=torch.int64, device=device)
    rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device)
    next_observations = torch.as_tensor(
        np.asarray(next_observations), dtype=torch.float32, device=device
    )
    terminated = torch.as_tensor(terminated, dtype=torch.float32, device=device)

    values = critic(observations).squeeze(-1)
    with torch.no_grad():
        next_values = critic(next_observations).squeeze(-1)
        returns = torch.empty_like(rewards)
        for index in reversed(range(len(rewards))):
            if index == len(rewards) - 1 or episode_ends[index]:
                bootstrapped_return = (
                    (1.0 - terminated[index]) * next_values[index]
                )
            bootstrapped_return = (
                rewards[index] + GAMMA * bootstrapped_return
            )
            returns[index] = bootstrapped_return
        advantages = returns - values
        normalized_advantages = (advantages - advantages.mean()) / (
            advantages.std(unbiased=False) + 1e-8
        )

    distribution = action_distribution(observations)
    policy_loss = -(
        distribution.log_prob(actions) * normalized_advantages
    ).mean()
    entropy = distribution.entropy().mean()
    actor_loss = policy_loss - ENTROPY_COEFFICIENT * entropy
    value_loss = nn.functional.huber_loss(values, returns)

    actor_optimizer.zero_grad()
    actor_loss.backward()
    nn.utils.clip_grad_norm_(actor.parameters(), MAX_GRAD_NORM)
    actor_optimizer.step()

    critic_optimizer.zero_grad()
    value_loss.backward()
    nn.utils.clip_grad_norm_(critic.parameters(), MAX_GRAD_NORM)
    critic_optimizer.step()

    return {
        "policy_loss": policy_loss.item(),
        "value_loss": value_loss.item(),
        "advantage": advantages.mean().item(),
        "entropy": entropy.item(),
    }

## 3. Collect fixed-length on-policy rollouts

Unlike the [vanilla notebook](actor_critic.ipynb), which updates after every transition, this advanced version collects `N_STEPS` transitions with the current policy, computes their bootstrapped returns, averages the actor and critic losses into one update, and then discards them. A rollout can cross episode boundaries: termination resets the environment and records the completed return, but does not force an early update. The stored episode-boundary flag stops return accumulation across that reset. This keeps the batch size and update cadence stable as the policy improves and episodes grow longer. The final shorter rollout is also updated rather than discarded.

In [ ]:
def train(total_timesteps):
    episode_returns, metrics = [], []
    rollout = []
    episode_return = 0.0
    observation, _ = env.reset()

    for step in range(1, total_timesteps + 1):
        action = select_action(observation)
        next_observation, reward, terminated, truncated, _ = env.step(action)
        rollout.append(
            (
                observation,
                action,
                reward,
                next_observation,
                terminated,
                terminated or truncated,
            )
        )
        episode_return += reward

        if terminated or truncated:
            episode_returns.append(episode_return)
            episode_return = 0.0
            observation, _ = env.reset()
        else:
            observation = next_observation

        if len(rollout) == N_STEPS or step == total_timesteps:
            metrics.append(update(*zip(*rollout, strict=True)))
            rollout.clear()

        print(
            f"\rStep {step}/{total_timesteps} | "
            f"Episodes: {len(episode_returns)} | "
            f"Episode return: {episode_return:.1f}",
            end="",
        )

    env.close()
    return episode_returns, metrics


episode_returns, metrics = train(TOTAL_TIMESTEPS)
print(f"\nTrained for {len(episode_returns)} episodes and {len(metrics)} updates.")

## 4. Inspect learning

Episode return is the main performance measure. Actor and critic losses are useful diagnostics, but neither needs to decrease monotonically because the policy continually changes the data distribution and the critic's targets.

In [ ]:
returns = np.asarray(episode_returns)
window = min(10, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
axes[0].plot(returns, alpha=0.35, label="episode return")
axes[0].plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode mean",
)
axes[0].set(title=f"Actor-Critic on {ENV_ID}", xlabel="Episode", ylabel="Return")
axes[0].legend()
axes[1].plot([item["value_loss"] for item in metrics], alpha=0.7)
axes[1].set(title="Value loss", xlabel="Update", ylabel="Huber loss")
axes[2].plot([item["policy_loss"] for item in metrics], alpha=0.7)
axes[2].set(title="Policy loss", xlabel="Update", ylabel="Loss")
axes[3].plot([item["entropy"] for item in metrics], alpha=0.7)
axes[3].set(title="Policy entropy", xlabel="Update", ylabel="Entropy")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Evaluate the modal policy

Training must remain stochastic, but evaluation can select the highest-probability action to measure the learned policy without sampling noise. Keep evaluation in a separate environment so it cannot disturb training state.

In [ ]:
env = gym.make(ENV_ID, render_mode="human")
episode_returns = []

for episode in range(5):
    observation, _ = env.reset()
    episode_return = 0.0
    for step in range(1000):
        action = select_action(observation, deterministic=True)
        observation, reward, terminated, truncated, _ = env.step(action)
        episode_return += reward
        print(
            f"Episode {episode + 1}: step={step + 1}, "
            f"return={episode_return:.1f}",
            end="\r",
        )
        if terminated or truncated:
            break
    episode_returns.append(episode_return)
    print()

env.close()
print(f"Mean return: {np.mean(episode_returns):.1f} +/- {np.std(episode_returns):.1f}")